# Stage 4: Development Grid Evaluation & Candidate Selection

**Milestone A2**: Knowledge Base Construction, Chunking, Embedding Models, & Vector Search.

### Protocol:
1. Evaluates a full **5 x 5 Factorial Grid** (5 Embedding Models x 5 Chunking Strategies) across the complete **1,034-page historical medical book** (364,824 words).
2. Grounded on **80 Development Queries** (60 single-page + 20 multi-page) spanning all 10 deciles of the book.
3. Applies the **Explainability Selection Rule** (Maximize Dev Unique-Page Recall@5 -> MRR@10 tiebreaker -> Latency/Size).
4. Locks the winning stack in `candidate-lock.json` and emits `stage4-dev-selection.zip`.


### 1. Environment & Offline Wheel Installation

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Enforce offline mode when running on Kaggle
if Path("/kaggle/working").is_dir():
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    INPUT_ROOT = Path("/kaggle/input")
    OUT_DIR = Path("/kaggle/working/dev_output")
else:
    INPUT_ROOT = Path("extras/indexing-benchmarks")
    OUT_DIR = Path("results/dev_output")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Install offline wheels if present
all_wheels = list(INPUT_ROOT.rglob("*.whl")) if INPUT_ROOT.is_dir() else []
if all_wheels:
    print(f"Installing {len(all_wheels)} offline wheels...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(all_wheels[0].parent), *[str(w) for w in all_wheels]], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("Offline wheels installed successfully.")

### 2. Module Discovery & Data Loading

In [ ]:
# Add code directory to path
for p in [Path("code"), Path("../code"), Path("extras/indexing-benchmarks/code"), *INPUT_ROOT.rglob("code")]:
    if p.is_dir() and (p / "corpus.py").is_file():
        if str(p.resolve()) not in sys.path:
            sys.path.insert(0, str(p.resolve()))
        print(f"Loaded benchmark package from: {p.resolve()}")
        break

from corpus import load_canonical_corpus
from queries import load_retrieval_queries
from chunking import build_chunk_suites
from models import discover_candidate_models, EmbeddingModelAdapter
from evaluation import evaluate_retrieval_suite

# Load 1,034-page corpus and 110-query suite
pages = load_canonical_corpus(INPUT_ROOT)
all_queries = load_retrieval_queries(INPUT_ROOT)
dev_queries = [q for q in all_queries if q.split == "dev"]

print("\n--- Canonical Corpus Statistics ---")
print(f"Total Document Pages: {len(pages):,}")
print(f"Nonempty Text Pages:  {sum(1 for p in pages if p.word_count > 0):,}")
print(f"Total Corpus Words:   {sum(p.word_count for p in pages):,}")
print(f"Development Queries:  {len(dev_queries)} (60 single-page + 20 multi-page)")

assert len(pages) == 1034, f"Corpus count error: expected 1034, got {len(pages)}"
assert len(dev_queries) == 80, f"Query count error: expected 30, got {len(dev_queries)}"


### 3. Build 5 Candidate Chunking Suites

In [ ]:
import numpy as np

print("Generating 5 whitespace word chunking suites...")
chunk_suites = build_chunk_suites(pages)
for name, c_list in chunk_suites.items():
    avg_words = np.mean([c.word_count for c in c_list])
    print(f" - {name:<24}: {len(c_list):>5} chunks (avg {avg_words:.1f} words)")

assert len(chunk_suites) == 5, f"Expected 5 chunking suites, got {len(chunk_suites)}"

### 4. Discover Candidate Embedding Models

In [ ]:
discovered = discover_candidate_models([INPUT_ROOT, Path(".")])
print(f"Discovered {len(discovered)} Candidate Embedding Models:")
for c_id, (_, m_path) in discovered.items():
    print(f" - {c_id:<24} -> {m_path}")

assert len(discovered) == 5, f"Expected 5 candidate models, got {len(discovered)}"

### 5. Execute 5x5 Factorial Grid on Development Set

In [ ]:
import torch
import json
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Execution Device: {device.upper()}")
print("\n" + "=" * 130)
head = f"{'Model':<22} {'Chunking Strategy':<24} {'Chunks':<7} {'R@1':<8} {'R@5':<8} {'MRR@10':<8} {'Span@5':<8} {'Multi-Cov':<10} {'All-Found':<10} {'95% CI (R@5)':<16}"
print(head)
print("-" * 130)

grid_results = []
all_dev_logs = []

for c_id, (_, m_path) in discovered.items():
    adapter = EmbeddingModelAdapter(m_path, canonical_id=c_id, device=device)
    for s_name, c_list in chunk_suites.items():
        is_pc = (s_name == "parent_child_128_512")
        metrics, logs = evaluate_retrieval_suite(
            adapter, c_list, dev_queries, top_k=10, is_parent_child=is_pc
        )
        grid_results.append(metrics)
        all_dev_logs.extend(logs)

        r1 = f"{metrics['single_page_recall@1']:.3f}"
        r5 = f"{metrics['single_page_recall@5']:.3f}"
        mrr = f"{metrics['single_page_mrr@10']:.4f}"
        span5 = f"{metrics['single_page_span_containment@5']:.3f}"
        mcov = f"{metrics['multi_page_coverage@10']:.3f}"
        mall = f"{metrics['multi_page_all_found@10']:.3f}"
        ci_str = f"[{metrics['recall@5_ci_95'][0]:.2f}, {metrics['recall@5_ci_95'][1]:.2f}]"
        print(f"{metrics['canonical_model_id']:<22} {s_name:<24} {len(c_list):<7} {r1:<8} {r5:<8} {mrr:<8} {span5:<8} {mcov:<10} {mall:<10} {ci_str:<16}")

print("=" * 130)
assert len(grid_results) == 25, f"Incomplete run: expected 25 cells, got {len(grid_results)}"

### 6. Lock Winning Candidate Stack (Explainability Rule)

In [ ]:
import pandas as pd

# Sort by Unique-Page Recall@5, tie-break by MRR@10, then Multi-Page Coverage
grid_results.sort(
    key=lambda x: (x["single_page_recall@5"], x["single_page_mrr@10"], x["multi_page_coverage@10"]),
    reverse=True,
)
winner = grid_results[0]

print("\n👑 WINNING CANDIDATE STACK LOCKED:")
print(f" - Canonical Model ID:   {winner['canonical_model_id']} ({winner['dimension']}-d)")
print(f" - Chunking Strategy:    {winner['strategy']} ({winner['total_chunks']} chunks, avg {winner['avg_words']} words)")
print(f" - Dev Unique Recall@5:  {winner['single_page_recall@5']:.4f} (95% CI: {winner['recall@5_ci_95']})")
print(f" - Dev MRR@10:           {winner['single_page_mrr@10']:.4f}")
print(f" - Dev Multi-Page Cov:   {winner['multi_page_coverage@10']:.4f}")

df_summary = pd.DataFrame(grid_results)
df_display = df_summary[["canonical_model_id", "strategy", "total_chunks", "single_page_recall@1", "single_page_recall@5", "single_page_mrr@10", "multi_page_coverage@10", "single_query_latency_ms"]]
display(df_display.head(10))

### 7. Emit Stage 4 Development Artifacts & ZIP

In [ ]:
import zipfile

corpus_stats = {
    "pdf_total_pages": len(pages),
    "nonempty_indexed_pages": sum(1 for p in pages if p.word_count > 0),
    "ocr_missing_unobserved_pages": sum(1 for p in pages if p.ocr_source == "ocr_missing_unobserved"),
    "ocr_empty_illustration_only_pages": sum(1 for p in pages if p.ocr_source == "ocr_empty_illustration_only"),
    "total_corpus_words": sum(p.word_count for p in pages),
}

query_audit = {
    "total_queries": len(all_queries),
    "dev_queries": len(dev_queries),
    "dev_single_page": sum(1 for q in dev_queries if q.type == "single_page"),
    "dev_multi_page": sum(1 for q in dev_queries if q.type == "multi_page"),
}

candidate_lock = {
    "winning_model_id": winner["canonical_model_id"],
    "resolved_model_path": winner["resolved_model_path"],
    "winning_chunk_strategy": winner["strategy"],
    "dimension": winner["dimension"],
    "dev_recall@5": winner["single_page_recall@5"],
    "dev_mrr@10": winner["single_page_mrr@10"],
    "dev_recall@5_ci_95": winner["recall@5_ci_95"],
}

run_env = {
    "python_version": sys.version,
    "device": device,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}

manifest = {
    "stage": "stage4-dev-selection",
    "version": "2.0-reproduced",
    "grid_cells_evaluated": len(grid_results),
    "timestamp": run_env["timestamp"],
}

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(OUT_DIR / "corpus-stats.json").write_text(json.dumps(corpus_stats, indent=2), encoding="utf-8")
(OUT_DIR / "query-audit.json").write_text(json.dumps(query_audit, indent=2), encoding="utf-8")
(OUT_DIR / "dev-grid-results.json").write_text(json.dumps(grid_results, indent=2), encoding="utf-8")
(OUT_DIR / "candidate-lock.json").write_text(json.dumps(candidate_lock, indent=2), encoding="utf-8")
(OUT_DIR / "run-environment.json").write_text(json.dumps(run_env, indent=2), encoding="utf-8")

with open(OUT_DIR / "dev-per-query.jsonl", "w", encoding="utf-8") as f:
    for log in all_dev_logs:
        f.write(json.dumps(log) + "\n")

zip_path = OUT_DIR / "stage4-dev-selection.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in [
        "manifest.json",
        "corpus-stats.json",
        "query-audit.json",
        "dev-grid-results.json",
        "dev-per-query.jsonl",
        "candidate-lock.json",
        "run-environment.json",
    ]:
        zf.write(OUT_DIR / fname, arcname=fname)

print(f"\nCreated Development Selection Package: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")